# CRML Fuzzing Harness Demo

Demonstrates the `FuzzHarness` against the **fluid storage** requirement model.
The harness compiles two CRML models to Modelica, builds a single
`ComparisonHarness` model that runs both implementations with the
same parametric mock, and checks for output divergence inside the
simulation (no post-hoc time-grid alignment needed).

**Sections**
1. Build & start JVM
2. Initialise harness
3. Sanity check — reference vs reference (expect all PASS)
4. Broken candidate — AND→OR logic error (expect targeted FAIL)
5. Debug a single failing parameter vector

## 1  Build & start JVM

In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parent  # CRML repo root
from experiments.gradle_jvm import GradleJvm

In [2]:
jvm = GradleJvm(
    project_path=ROOT,
    subproject="experiments",
    subproject_dir="submodules/experiments",
)
jvm.build()
jvm.start()

Running: /home/ubuntu/crml/vol/CRML/gradlew experiments:shadowJar  (cwd=/home/ubuntu/crml/vol/CRML)
Starting a Gradle Daemon, 2 incompatible and 2 stopped Daemons could not be reused, use --status for details
> Task :language:generateGrammarSource UP-TO-DATE
> Task :language:compileJava UP-TO-DATE
> Task :util:generateGrammarSource NO-SOURCE
> Task :util:compileJava UP-TO-DATE
> Task :compiler:compileJava UP-TO-DATE
> Task :compiler:processResources UP-TO-DATE
> Task :compiler:classes UP-TO-DATE
> Task :compiler:jar UP-TO-DATE
> Task :experiments:compileJava UP-TO-DATE
> Task :experiments:processResources UP-TO-DATE
> Task :experiments:classes UP-TO-DATE
> Task :language:processResources UP-TO-DATE
> Task :language:classes UP-TO-DATE
> Task :language:jar UP-TO-DATE
> Task :util:processResources NO-SOURCE
> Task :util:classes UP-TO-DATE
> Task :util:jar UP-TO-DATE
> Task :experiments:shadowJar

BUILD SUCCESSFUL in 11s
12 actionable tasks: 1 executed, 11 up-to-date
Consider enabling conf

## 2  Initialise harness

In [3]:
from experiments.harness import CRMLCompiler, FuzzHarness
from experiments.domains import CRMLTOMODELICA_PATH, FLUID_STORAGE_REF_CRML, FLUID_STORAGE_DOMAIN

compiler = CRMLCompiler()
harness  = FuzzHarness(FLUID_STORAGE_DOMAIN, compiler, CRMLTOMODELICA_PATH)

ref_crml = FLUID_STORAGE_REF_CRML.read_text()
print(f"Reference CRML: {len(ref_crml)} chars, {ref_crml.count(chr(10))} lines")
print(ref_crml)

Reference CRML: 362 chars, 16 lines
model SafeLevelRequirement is {

    class TankModel is {
        Real waterLevel is external;
        Real maxVolume is external;

        // Level stays within 80% of tank volume
        Boolean levelSafe is waterLevel < (0.8 * maxVolume);
    };

    TankModel tank1;
    TankModel tank2;

    Boolean tanksAreSafe is tank1.levelSafe and tank2.levelSafe;

};



## 3  Sanity check — reference vs reference

Both candidate and reference are the same source.  Every simulation run
must produce identical output signals, so the harness should report
**zero failures** across all named scenarios and random fuzz iterations.

In [4]:
result_sanity = harness.run(
    candidate_crml=ref_crml,
    reference_crml=ref_crml,
    n_iters=20,
    seed=42,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_sanity.summary())

[harness] Compiling candidate...
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[harness] Compiling reference...
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[harness] Building ComparisonHarness... /home/ubuntu/crml/vol/data
[runner] Setup... /home/ubuntu/crml/vol/data
[harness] Fuzz 20/20...
PASS  26/26 runs matched


Expected output:
```
PASS  26/26 runs matched
```
(6 named scenarios + 20 random iterations)

## 4  Broken candidate — AND→OR logic error

The reference requires **both** tanks to be within 80 % of their capacity:
```
Boolean tanksAreSafe is tank1.levelSafe and tank2.levelSafe;
```
The broken candidate replaces `and` with `or`, meaning the system is
declared safe whenever **either** tank is fine — violating fail-safe semantics.

This only diverges when exactly one tank overflows, e.g. the `tank1_overflow`
scenario (tank 1 at 90 % > 80 % threshold, tank 2 nominal at 40 %):

| model     | logic | tank1 safe? | tank2 safe? | tanksAreSafe |
|-----------|-------|-------------|-------------|--------------|
| reference | AND   | **false**   | true        | **false**    |
| broken    | OR    | **false**   | true        | **true**     |

The `both_overflow` scenario (both tanks fail simultaneously) makes both
models output `false`, so **no mismatch** there — demonstrating that not
every fault exposes the bug.

In [5]:
# Change AND to OR in the top-level safety requirement
broken_crml = ref_crml.replace(
    "Boolean tanksAreSafe is tank1.levelSafe and tank2.levelSafe;",
    "Boolean tanksAreSafe is tank1.levelSafe or tank2.levelSafe;",
)

assert broken_crml != ref_crml, "String replacement had no effect — check the source"
print("Broken CRML prepared.")

Broken CRML prepared.


In [6]:
result_broken = harness.run(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    n_iters=20,
    seed=42,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_broken.summary())

[harness] Compiling candidate...
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : or og_op : or
[harness] Compiling reference...
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[harness] Building ComparisonHarness... /home/ubuntu/crml/vol/data
[runner] Setup... /home/ubuntu/crml/vol/data
[harness] Fuzz 20/20...
FAIL  19/26 runs diverged
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank1_overflow'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank2_overflow'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank1_brief'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'varied_volumes'}
  signals: ['tanksAreSafe']
  params:  {'maxVolume1': 2.7740987964651307, 'maxVolume2': 1.3947694280899183, 'tank1_fault_start': 91.7412220904825, 'tank1_recovery': 1000000000.0, 'tank2_fault_start': 131.414568501674, 'tank2_recovery': 1000000000.0}
  signals: ['tanksAreSafe']
 

Expected output (at minimum):
```
FAIL  19/26 runs diverged
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank1_overflow'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank2_overflow'}
  ...
```
The `nominal` and `both_overflow` (simultaneous) scenarios match — not every
fault combination exposes the AND→OR bug.  Random iterations where exactly
one tank overflows are also flagged.

## 5  Debug — re-run a single failing vector

`run_scenario` compiles and builds once for a single parameter dict,
useful for reproducing a specific failure reported by `run()`.

In [7]:
# Re-run the exact scenario that exposed the bug:
# tank1 overflows (level = 90% of maxVolume > 80% threshold),
# tank2 stays nominal (40% < 80%) — only AND vs OR differs here.
debug_params = {
    "tank1_fault_start": 30.0,
    # tank1_recovery left at default (1e9) — stays faulted
}

result_debug = harness.run_scenario(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    params=debug_params,
    verbose=True,
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print(result_debug.summary())

Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : or og_op : or
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[harness] Building ComparisonHarness...
[runner] Setup... /home/ubuntu/crml/vol/data
FAIL  1/1 runs diverged
  signals: ['tanksAreSafe']
  params:  {'tank1_fault_start': 30.0}


In [8]:
# Confirm simultaneous both-tank overflow does NOT trigger a mismatch:
# both tanks fail at t=20 → AND = false, OR = false → they agree.
result_both = harness.run_scenario(
    candidate_crml=broken_crml,
    reference_crml=ref_crml,
    params={"tank1_fault_start": 20.0, "tank2_fault_start": 20.0},
    work_dir=Path("~/crml/vol/data").expanduser(),
    keep=True,
)

print("Both overflow:", result_both.summary())

Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : or og_op : or
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[runner] Setup... /home/ubuntu/crml/vol/data
Both overflow: PASS  1/1 runs matched


## 6  Semantic Analysis — mapping-driven harness run

Generated CRML models use different variable names for the same requirements
(e.g. `R1` instead of `R1_T`).  We bridge this with a per-file JSON mapping
produced by Claude (see `generated/MATCHING_PROMPT.md`).

**Workflow:**
1. For each generated `.crml`, run the prompt in `MATCHING_PROMPT.md` to produce `<name>_mapping.json`.
2. `RequirementMapping.load()` reads the JSON.
3. `mapping.apply_to_domain()` returns a `DomainSpec` with only the matched outputs,
   each `OutputSignal` carrying the generated variable name as `candidate_name`.
4. Run `FuzzHarness` normally; catch `OMCBuildError` for structurally incompatible models.

In [14]:
from experiments.harness import FuzzHarness, OMCBuildError, RequirementMapping
from experiments.domains import CRMLTOMODELICA_PATH, FLUID_STORAGE_REF_CRML, FLUID_STORAGE_DOMAIN

# Path helpers
GEN_ROOT = Path("generated__old")
ref_crml = FLUID_STORAGE_REF_CRML.read_text()

def mapping_path(crml_path: Path) -> Path:
    return crml_path.with_name(crml_path.stem + "_mapping.json")

### 6a  Single model — inspect mapping and run

In [15]:
# Edit these two paths to analyse any generated model.
crml_file   = GEN_ROOT / "claude" / "SRI_temp_k1.crml"
mapping_file = mapping_path(crml_file)   # generated/claude/SRI_temp_k1_mapping.json

mapping = RequirementMapping.load(mapping_file)
print(mapping.report())

Requirement mapping:
  R1_T                 → R1
  R2_T                 → R2
  R_T                  → (missing)
  R_speed_all          → (missing)
  R_flow_all           → (missing)

2/5 requirements matched.


In [16]:
adapted_domain  = mapping.apply_to_domain(FLUID_STORAGE_DOMAIN)
adapted_harness = FuzzHarness(adapted_domain, compiler, CRMLTOMODELICA_PATH)
gen_crml        = crml_file.read_text()

try:
    result = adapted_harness.run(
        candidate_crml=gen_crml,
        reference_crml=ref_crml,
        n_iters=20,
        seed=42,
        verbose=True,
        work_dir=Path("~/crml/vol/data").expanduser(),
        keep=True,
    )
    print(result.summary())
except OMCBuildError as e:
    print(f"Build failed (structural incompatibility or missing externals):\n{e}")

[harness] Compiling candidate...
Category: null og_op : not og_op : not
[harness] Compiling reference...
Category: null og_op : and og_op : and
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b2false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(CRMLtoModelica.Functions.not4( b2), CRMLtoModelica.Functions.not4( b1))false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : and og_op : and
Applying operator: or b2 b1 

Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(b2, b1)false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Applying operator: or b2 notb1 

Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : filter 

Added category increasing_int

Added category increasing_real

Added category varying1

Added category varying2



[harness] Fuzz 20/20...
FAIL  19/26 runs diverged
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank1_overflow'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank2_overflow'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'tank1_brief'}
  signals: ['tanksAreSafe']
  params:  {'scenario': 'varied_volumes'}
  signals: ['tanksAreSafe']
  params:  {'maxVolume1': 2.7740987964651307, 'maxVolume2': 1.3947694280899183, 'tank1_fault_start': 91.7412220904825, 'tank1_recovery': 1000000000.0, 'tank2_fault_start': 131.414568501674, 'tank2_recovery': 1000000000.0}
  signals: ['tanksAreSafe']
  params:  {'maxVolume1': 3.6416272774470206, 'maxVolume2': 1.199657749153017, 'tank1_fault_start': 60.48918176689401, 'tank1_recovery': 1000000000.0, 'tank2_fault_start': 1000000000.0, 'tank2_recovery': 116.59616501870258}
  signals: ['tanksAreSafe']
  params:  {'maxVolume1': 1.4427816384716947, 'maxVolume2': 4.714945644706223, 'tank1_fault_start': 101.36965085004823, 'tank1_recovery': 10000

### 6b  Batch — all models with mapping files

In [17]:
results_table = []

for crml_file in sorted(GEN_ROOT.rglob("*.crml")):
    mp = mapping_path(crml_file)
    if not mp.exists():
        results_table.append((crml_file, "no mapping", None))
        continue

    mapping     = RequirementMapping.load(mp)
    adapted     = mapping.apply_to_domain(FLUID_STORAGE_DOMAIN)
    gen_crml    = crml_file.read_text()
    h           = FuzzHarness(adapted, compiler, CRMLTOMODELICA_PATH)

    try:
        r = h.run(
            candidate_crml=gen_crml,
            reference_crml=ref_crml,
            n_iters=20,
            seed=42,
            verbose=False,
        )
        results_table.append((crml_file, "ok", r))
    except OMCBuildError:
        results_table.append((crml_file, "build_error", None))

print(f"{'Model':<50s} {'Status':<12s} {'Result'}")
print("-" * 80)
for path, status, r in results_table:
    label = str(path.relative_to(GEN_ROOT))
    verdict = r.summary().split("\n")[0] if r else "-"
    print(f"{label:<50s} {status:<12s} {verdict}")

Category: null og_op : <= og_op : <=
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[runner] Setup... /tmp/crml_harness_w53kneqw
OMC errors:
  [/tmp/crml_harness_w53kneqw/crml_harness/ComparisonHarness.mo:19:18-21:2:writable] Error: No viable alternative near token: ;
Category: null og_op : <= og_op : <=
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[runner] Setup... /tmp/crml_harness_13hrm37f
OMC errors:
  [/tmp/crml_harness_13hrm37f/crml_harness/ComparisonHarness.mo:19:18-21:2:writable] Error: No viable alternative near token: ;
Category: null og_op : <= og_op : <=
Category: null og_op : < og_op : <
Category: null og_op : * og_op : *
Category: null og_op : and og_op : and
[runner] Setup... /tmp/crml_harness_uqhsn5sg
OMC errors:
  [/tmp/crml_harness_uqhsn5sg/crml_harness/ComparisonHarness.mo:19:18-21:2:writable] Error: No viable alternative near token: ;
Categor

Added category increasing_int

Added category increasing_real

Added category varying1

Added category varying2



OMC errors:
  [/tmp/crml_harness_lsg5g_fy/crml_harness/ComparisonHarness.mo:19:18-21:2:writable] Error: No viable alternative near token: ;
Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b2false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(CRMLtoModelica.Functions.not4( b2), CRMLtoModelica.Functions.not4( b1))false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : and og_op : and
Applying operator: or b2 b1 

Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(b2, b1)false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Applying operator: or b2 notb1 

Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG 

Added category increasing_int

Added category increasing_real

Added category varying1

Added category varying2

Added category increasing_int



OMC errors:
  [/tmp/crml_harness_1dh34hnw/crml_harness/ComparisonHarness.mo:19:18-21:2:writable] Error: No viable alternative near token: ;
Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b2false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(CRMLtoModelica.Functions.not4( b2), CRMLtoModelica.Functions.not4( b1))false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Category: null og_op : and og_op : and
Applying operator: or b2 b1 

Category: null og_op : not og_op : not
Category: null og_op : and og_op : and
OP DEBUG: not  Boolean CRMLtoModelica.Functions.and4(b2, b1)false
SIG NAME: CRMLtoModelica.Functions.not4 [false]
Applying operator: or b2 notb1 

Category: null og_op : not og_op : not
OP DEBUG: not  Boolean b1false
SIG 

Added category increasing_real

Added category varying1

Added category varying2



org.antlr.v4.runtime.misc.ParseCancellationException: org.antlr.v4.runtime.misc.ParseCancellationException: Built in operator undefined : integrate on Boolean and Periods


## Shutdown

In [18]:
jvm.shutdown()

JVM shut down.
